## Data Download

In [1]:
import gdown
import os
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config

FILE_ID     = "1wtH8dGFqvNco3OcSd5V0sdfJ_NGmiOdB"
URL         = f"https://drive.google.com/uc?id={FILE_ID}"
DATA_DIR    = config.DATA_DIR
RAW_DIR     = DATA_DIR / "raw"
RAW_PATH    = config.RAW_PATH
TRAIN_PATH  = config.TRAIN_PATH
TEST_PATH   = config.TEST_PATH
SPLIT_YEAR  = config.SPLIT_YEAR

In [2]:
os.makedirs(RAW_DIR, exist_ok=True)
gdown.download(
    URL,
    str(RAW_DIR / "brk_metrics.parquet"),
    quiet=False,
    resume=True 
)

Skipping already downloaded file C:\Users\Shruti\OneDrive\Desktop\UBC\DSCI591\DevBranch\AdaptiveSats\data\raw\brk_metrics.parquet


'C:\\Users\\Shruti\\OneDrive\\Desktop\\UBC\\DSCI591\\DevBranch\\AdaptiveSats\\data\\raw\\brk_metrics.parquet'

## Data Splitting

In [3]:
import polars as pl

full_df = pl.read_parquet(RAW_PATH)
os.makedirs(DATA_DIR/"processed", exist_ok=True)

### Split year

`SPLIT_YEAR` defines the temporal boundary between training and test data.
Records with `day_utc` before this year form the training set; records from this year onward form the held-out test set.  2024 was chosen because it is the most recent complete calendar year available in the dataset, giving the model a realistic out-of-sample evaluation window while retaining the bulk of historical data for training.

In [4]:
train = full_df.filter(pl.col("day_utc").dt.year() < SPLIT_YEAR)
test  = full_df.filter(pl.col("day_utc").dt.year() >= SPLIT_YEAR)

train.write_parquet(TRAIN_PATH, compression="zstd")
test.write_parquet(TEST_PATH, compression="zstd")